## Классификация текстов с использованием предобученных языковых моделей.

В данном задании вам предстоит обратиться к задаче классификации текстов и решить ее с использованием предобученной модели BERT.

In [1]:
import json
# do not change the code in the block below
# __________start of block__________
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve

%matplotlib inline
# __________end of block__________

In [11]:
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

Обратимся к набору данных SST-2. Holdout часть данных (которая понадобится вам для посылки) доступна по ссылке ниже.

In [3]:
# do not change the code in the block below
# __________start of block__________

!wget https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json
# __________end of block__________

--2025-03-25 16:07:04--  https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json
185.199.109.133, 185.199.110.133, 185.199.108.133, ...content.com)… 
соединение установлено.busercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... 
HTTP-запрос отправлен. Ожидание ответа… 200 OK
Длина: 51581 (50K) [text/plain]
Сохранение в: «texts_holdout.json.6»

texts_holdout.json. 100%[===================>]  50,37K  --.-KB/s    за 0,04s   

2025-03-25 16:07:05 (1,09 MB/s) - «texts_holdout.json.6» сохранён [51581/51581]



In [4]:
# do not change the code in the block below
# __________start of block__________
df = pd.read_csv(
    "https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv",
    delimiter="\t",
    header=None,
)
texts_train = df[0].values[:5000]
y_train = df[1].values[:5000]
texts_test = df[0].values[5000:]
y_test = df[1].values[5000:]
with open("texts_holdout.json") as iofile:
    texts_holdout = json.load(iofile)
# __________end of block__________

In [33]:
texts_train = list(texts_train)
y_train = list(y_train)
texts_test = list(texts_test)
y_test = list(y_test)

# Для отправки в контест - нули просто, чтобы засунуть в TextDataset
texts_holdout = list(texts_holdout)
y_holdout = [0] * len(texts_holdout)

Весь остальной код предстоит написать вам.

Для успешной сдачи на максимальный балл необходимо добиться хотя бы __84.5% accuracy на тестовой части выборки__.

In [6]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device

device(type='mps')

In [35]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_text(text):
    return tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt")

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenize_text(texts)
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {key: val[idx].to(device) for key, val in self.encodings.items()}, self.labels[idx].to(device)


train_dataset = TextDataset(texts_train, y_train)
test_dataset = TextDataset(texts_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Для отправки в контест
holdout_dataset = TextDataset(texts_holdout, y_holdout)
holdout_loader = DataLoader(holdout_dataset, batch_size=16, shuffle=False)

In [12]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
lr_scheduler = ReduceLROnPlateau(optimizer, patience=35)

In [14]:
epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader):
        inputs, labels = batch
        optimizer.zero_grad()
        outputs = model(**inputs)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        lr_scheduler.step(loss.item())
        
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader)}")

100%|█████████████████████████████████████████| 313/313 [01:47<00:00,  2.91it/s]


Epoch 1: Loss = 0.38061913507529344


100%|█████████████████████████████████████████| 313/313 [01:46<00:00,  2.94it/s]


Epoch 2: Loss = 0.2625303609421649


100%|█████████████████████████████████████████| 313/313 [01:49<00:00,  2.85it/s]

Epoch 3: Loss = 0.2584763058363058


In [16]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in tqdm(test_loader):
        inputs, labels = batch
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)
        
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

100%|█████████████████████████████████████████| 120/120 [00:09<00:00, 12.28it/s]

Test Accuracy: 0.8917


In [36]:
model.eval()

with torch.no_grad():
    train_preds = np.empty(0)
    test_preds = np.empty(0)
    holdout_preds = np.empty(0)
    
    for batch in tqdm(train_loader):
        inputs, labels = batch
        outputs = model(**inputs)
        train_preds = np.append(train_preds, outputs.logits[:, 1].cpu().detach().numpy())

    for batch in tqdm(test_loader):
        inputs, labels = batch
        outputs = model(**inputs)
        test_preds = np.append(test_preds, outputs.logits[:, 1].cpu().detach().numpy())

    for batch in tqdm(holdout_loader):
        inputs, labels = batch
        outputs = model(**inputs)
        holdout_preds = np.append(holdout_preds, outputs.logits[:, 1].cpu().detach().numpy())

100%|███████████████████████████████████████████| 32/32 [00:02<00:00, 10.68it/s]


In [37]:
train_preds.shape, test_preds.shape, holdout_preds.shape

((5000,), (1920,), (500,))

#### Сдача взадания в контест
Сохраните в словарь `out_dict` вероятности принадлежности к первому (положительному) классу

In [39]:
out_dict = {
    'train': list(train_preds),
    'test': list(test_preds),
    'holdout': list(holdout_preds)
}

Несколько `assert`'ов для проверки вашей посылки:

In [42]:
assert isinstance(out_dict["train"], list), "Object must be a list of floats"
assert isinstance(out_dict["train"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["train"]) == 5000
), "The predicted probas list length does not match the train set size"

assert isinstance(out_dict["test"], list), "Object must be a list of floats"
assert isinstance(out_dict["test"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["test"]) == 1920
), "The predicted probas list length does not match the test set size"

assert isinstance(out_dict["holdout"], list), "Object must be a list of floats"
assert isinstance(out_dict["holdout"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["holdout"]) == 500
), "The predicted probas list length does not match the holdout set size"

Запустите код ниже для генерации посылки.

In [43]:
# do not change the code in the block below
# __________start of block__________
FILENAME = "submission_dict_hw_text_classification_with_bert.json"

with open(FILENAME, "w") as iofile:
    json.dump(out_dict, iofile)
print(f"File saved to `{FILENAME}`")
# __________end of block__________

File saved to `submission_dict_hw_text_classification_with_bert.json`


На этом задание завершено. Поздравляем!

In [45]:
save_path = "./bert_classification_model"

# Save model and tokenizer
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

('./bert_classification_model/tokenizer_config.json',
 './bert_classification_model/special_tokens_map.json',
 './bert_classification_model/vocab.txt',
 './bert_classification_model/added_tokens.json')